# Creator Shortlisting - intelligent creator discovery + budget optimization

**NYU SPS x Google Hackathon | Track 2 (Product & Engineering)**

A brand enters a single brief and the tool runs the whole flow automatically:

```
brief (keywords / audience / competitors / budget)
  -> (1) auto-discover candidates (search API by keyword)
  -> (2) auto-fetch data (channels/videos API)
  -> (3) auto-score (quantitative metrics + LLM)
  -> (4) auto-rank + hard cutoffs + rationale
  -> (5) budget optimization (pricing -> value-per-dollar -> 0/1 knapsack)
  -> output: ranked table + "optimal set for a given $X"
```

**Mode**: with a YouTube API key = LIVE (real data); without = DEMO (built-in samples).
LLM scoring uses Gemini when a key is present, otherwise a keyword/synonym fallback.

In [ ]:
import json
import config
import scoring
import llm
import pricing
import pipeline

print("LIVE mode" if config.YOUTUBE_API_KEY else "DEMO mode (no YouTube key)")
print("LLM:", "Gemini" if config.GEMINI_API_KEY else "keyword/synonym fallback")

## (1) Input brief (this is exactly what the frontend teammate fills in)

In [ ]:
campaign_input = {
    "brief": "Eco-friendly skincare brand, target women 18-34",
    "keywords": ["skincare", "clean beauty", "eco-friendly"],
    "competitors": ["BrandX"],
    "risk_topics": ["political", "controversial", "scandal"],
    "budget_cap": 5000,      # total budget ($)
    "target_k": 3,           # how many creators to sign
}
print(json.dumps(campaign_input, ensure_ascii=False, indent=2))

## (2) Auto-discover candidates + fetch real data

In [ ]:
if config.YOUTUBE_API_KEY:
    try:
        import youtube_api
        creators = youtube_api.get_creators(campaign_input)
        mode = "LIVE"
    except Exception as e:
        print(f"[warn] live API failed ({e}), falling back to sample data")
        import mock
        creators, mode = mock.get_creators(), "DEMO"
else:
    import mock
    creators, mode = mock.get_creators(), "DEMO"

print(f"Discovered {len(creators)} candidate channels (mode={mode})")
for c in creators[:5]:
    print(f"  - {c['handle']:<30} {c['subscriber_count']:>10,} subs | {len(c['videos'])} recent videos")

## (3-5) Run the full pipeline (score -> cutoff -> rank -> budget optimization)

In [ ]:
results = pipeline.run_pipeline(campaign_input, creators, mode=mode)

print(f"{len(results['creators'])} kept, {len(results['excluded'])} excluded by hard cutoff")
for e in results["excluded"]:
    print(f"  x {e['handle']:<28} {e['reason']}")
print("CPM benchmark assumption:", results["cpm_assumption"], "$/1k views")

## Ranking table (every score is decomposable = transparent)

In [ ]:
import pandas as pd

rows = []
for r in results["creators"]:
    s = r["scores"]
    rows.append({
        "rank": r["rank"], "handle": r["handle"], "subscribers": r["subscribers"],
        "total": r["total"],
        "content_match": s["content_match"], "engagement": s["engagement"],
        "growth": s["growth"], "stability": s["stability"],
        "audience_fit": s["audience_fit"], "consistency": s["consistency"],
        "avg_views": r["avg_views"], "estimated_cost": r["estimated_cost"],
        "value_per_dollar": r["value_per_dollar"], "risk": r["flags"]["risk"],
    })
df = pd.DataFrame(rows)
df.head(15)

## Budget optimization (0/1 knapsack)

In [ ]:
p = results["portfolio"]
if p:
    print(f"Within ${campaign_input['budget_cap']}, recommended {campaign_input['target_k']} creators:")
    print(f"  {', '.join(p['picks'])}")
    print(f"  total cost ${p['total_cost']:,.2f}")
    print(f"  expected views {p['expected_views']:,.0f} | expected engagements {p['expected_engagements']:,.0f}")
    print(f"  algorithm: {p['note']}")
else:
    print("no feasible combination within budget")

## Full `results` for the frontend (the interface contract, ready to hand off)

In [ ]:
print(json.dumps(results, ensure_ascii=False, indent=2, default=str))